In [1]:

import os
from crewai_tools import SerperDevTool, BrowserbaseLoadTool, EXASearchTool
from crewai import Agent
from crewai import Task
from crewai import LLM

from langchain_community.llms import Ollama


from crewai import Crew, Process

from crewai_tools import WebsiteSearchTool


In [2]:

import paramiko

print(   paramiko.__version__   )


5.0.0


In [3]:

'''

os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"

'''



'''
openai_llm = LLM(
    model="gpt-4",
    api_key="your-openai-api-key",
    api_base="https://api.openai.com/v1"
)
'''


'\nopenai_llm = LLM(\n    model="gpt-4",\n    api_key="your-openai-api-key",\n    api_base="https://api.openai.com/v1"\n)\n'

In [4]:

researcher_agent = Agent(
    role='Cyber Security Analyst',
    goal='Analyze and test cyber security in your corporation',
    backstory="""You're a cyber security specialist at a large company.
    You're responsible for analyzing security data and testing the company cyber security.""",
    verbose=False,
    ## llm=llm_llama
    ## llm=LLM( model="ollama/llama3.2", base_url="http://localhost:11434" )
    llm=LLM( model="ollama/qwen2.5:0.5b", base_url="http://localhost:11434" )
)


In [5]:

task_description      = "What is the linux command to ping 8.8.8.8"
task_expected_output  = "analysis or procedural steps for cyber security"

research_task = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_agent,
    ## tools=[ WebsiteSearchTool() ],
    async_execution=True
)


In [ ]:



my_crew = Crew(
    agents=[researcher_agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)




In [ ]:



result = my_crew.kickoff()
print(result)





## Cleaning of raw data from LLM


In [ ]:

res1 = result.raw


res1= res1.replace("\n", "")

res1


In [6]:

import paramiko


In [7]:

VM_IP = "127.0.0.1"   # replace with your VM's IP
my_port = 2222
USERNAME = "seed"
PASSWORD = "dees"


In [8]:

# This is what your CrewAI agent produced
## results = "ping -c 4 8.8.8.8"

## results = "ls -l"

results = "id"


In [9]:



ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())




In [10]:

ssh.connect(
    VM_IP,
    port=my_port,
    username=USERNAME,
    password=PASSWORD
)


In [11]:

stdin, stdout, stderr = ssh.exec_command(results)


In [12]:

print("COMMAND:")
print(results)

print("\nOUTPUT:")
print(stdout.read().decode())

print("ERRORS:")
print(stderr.read().decode())

ssh.close()


COMMAND:
id

OUTPUT:
uid=1000(seed) gid=1000(seed) groups=1000(seed),4(adm),24(cdrom),27(sudo),30(dip),46(plugdev),120(lpadmin),131(lxd),132(sambashare),136(docker),1003(datascience)

ERRORS:



In [13]:

import paramiko
import re

# ============================================================
# VM SETTINGS
# ============================================================

VM_IP = "127.0.0.1"       # CHANGE
VM_PORT = 2222                # CHANGE
USERNAME = "seed"    # CHANGE
PASSWORD = "dees"    # CHANGE


In [14]:

# ============================================================
# CONNECT TO UBUNTU VM
# ============================================================

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

ssh.connect(
    VM_IP,
    port=VM_PORT,
    username=USERNAME,
    password=PASSWORD
)

print("Connected to VM.")


Connected to VM.


In [15]:



# ============================================================
# 1. EXECUTE A COMMAND GENERATED BY THE LLM
# ============================================================

# Replace this with your CrewAI result
results = "ping -c 4 8.8.8.8"

# Clean up LLM output
command = results.replace("\n", " ").strip()

print("\nCOMMAND:")
print(command)

stdin, stdout, stderr = ssh.exec_command(command)

output = stdout.read().decode()
error = stderr.read().decode()

print("\nOUTPUT:")
print(output)

if error:
    print("\nERROR:")
    print(error)





COMMAND:
ping -c 4 8.8.8.8

OUTPUT:
PING 8.8.8.8 (8.8.8.8) 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=112 time=54.3 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=112 time=53.5 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=112 time=37.2 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=112 time=40.7 ms

--- 8.8.8.8 ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 3015ms
rtt min/avg/max/mdev = 37.205/46.423/54.259/7.583 ms



In [16]:

# Ubuntu versions differ:
# /var/log/auth.log is common when rsyslog is being used.
# journalctl works on systemd systems.

log_command = "journalctl -u ssh --no-pager -n 50"

stdin, stdout, stderr = ssh.exec_command(log_command)

log_data = stdout.read().decode()
log_error = stderr.read().decode()

print("\n================ AUTH LOG ================\n")
print(log_data)

if log_error:
    print(log_error)



================ AUTH LOG ================

-- Logs begin at Thu 2023-01-12 15:25:21 EST, end at Thu 2026-09-17 22:05:50 EDT. --
Sep 13 19:11:15 VM sshd[797]: Server listening on 0.0.0.0 port 22.
Sep 13 19:11:15 VM sshd[797]: Server listening on :: port 22.
Sep 13 19:11:15 VM systemd[1]: Started OpenBSD Secure Shell server.
-- Reboot --
Sep 13 19:42:00 VM systemd[1]: Starting OpenBSD Secure Shell server...
Sep 13 19:42:00 VM sshd[761]: Server listening on 0.0.0.0 port 22.
Sep 13 19:42:00 VM sshd[761]: Server listening on :: port 22.
Sep 13 19:42:00 VM systemd[1]: Started OpenBSD Secure Shell server.
Sep 13 19:42:54 VM sshd[3163]: Accepted password for seed from 10.0.2.2 port 65067 ssh2
Sep 13 19:42:54 VM sshd[3163]: pam_unix(sshd:session): session opened for user seed by (uid=0)
Sep 13 19:44:22 VM sshd[3321]: Accepted password for seed from 10.0.2.2 port 65070 ssh2
Sep 13 19:44:22 VM sshd[3321]: pam_unix(sshd:session): session opened for user seed by (uid=0)
Sep 13 19:45:08 VM sshd[34

In [17]:

prompt = f"""
You are a cybersecurity analyst.

Analyze the following SSH authentication log.

Identify:
1. Failed login attempts
2. Successful logins
3. Suspicious IP addresses
4. Anything that might indicate an attack

LOG:

{log_data}
"""

print("\n================ PROMPT FOR LLM ================\n")
print(prompt)



================ PROMPT FOR LLM ================


You are a cybersecurity analyst.

Analyze the following SSH authentication log.

Identify:
1. Failed login attempts
2. Successful logins
3. Suspicious IP addresses
4. Anything that might indicate an attack

LOG:

-- Logs begin at Thu 2023-01-12 15:25:21 EST, end at Thu 2026-09-17 22:05:50 EDT. --
Sep 13 19:11:15 VM sshd[797]: Server listening on 0.0.0.0 port 22.
Sep 13 19:11:15 VM sshd[797]: Server listening on :: port 22.
Sep 13 19:11:15 VM systemd[1]: Started OpenBSD Secure Shell server.
-- Reboot --
Sep 13 19:42:00 VM systemd[1]: Starting OpenBSD Secure Shell server...
Sep 13 19:42:00 VM sshd[761]: Server listening on 0.0.0.0 port 22.
Sep 13 19:42:00 VM sshd[761]: Server listening on :: port 22.
Sep 13 19:42:00 VM systemd[1]: Started OpenBSD Secure Shell server.
Sep 13 19:42:54 VM sshd[3163]: Accepted password for seed from 10.0.2.2 port 65067 ssh2
Sep 13 19:42:54 VM sshd[3163]: pam_unix(sshd:session): session opened for user seed 

In [18]:

## task_description      = "What is the linux command to ping 8.8.8.8"

task_description = prompt

task_expected_output  = "analysis or procedural steps for cyber security"

research_task = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_agent,
    ## tools=[ WebsiteSearchTool() ],
    async_execution=True
)


In [19]:

my_crew = Crew(
    agents=[researcher_agent],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True
)


In [20]:



result = my_crew.kickoff()
print(result)




# Agent: Cyber Security Analyst
## Task: 
You are a cybersecurity analyst.

Analyze the following SSH authentication log.

Identify:
1. Failed login attempts
2. Successful logins
3. Suspicious IP addresses
4. Anything that might indicate an attack

LOG:

-- Logs begin at Thu 2023-01-12 15:25:21 EST, end at Thu 2026-09-17 22:05:50 EDT. --
Sep 13 19:11:15 VM sshd[797]: Server listening on 0.0.0.0 port 22.
Sep 13 19:11:15 VM sshd[797]: Server listening on :: port 22.
Sep 13 19:11:15 VM systemd[1]: Started OpenBSD Secure Shell server.
-- Reboot --
Sep 13 19:42:00 VM systemd[1]: Starting OpenBSD Secure Shell server...
Sep 13 19:42:00 VM sshd[761]: Server listening on 0.0.0.0 port 22.
Sep 13 19:42:00 VM sshd[761]: Server listening on :: port 22.
Sep 13 19:42:00 VM systemd[1]: Started OpenBSD Secure Shell server.
Sep 13 19:42:54 VM sshd[3163]: Accepted password for seed from 10.0.2.2 port 65067 ssh2
Sep 13 19:42:54 VM sshd[3163]: pam_unix(sshd:session): session opened for user seed by (uid=0)

In [21]:

res1 = result.raw


res1= res1.replace("\n", "")

res1


"**The given network traffic does not exhibit any obvious security threats consistent with cyber security criteria. However, it's crucial to monitor and analyze further for potential lateral movement or unauthorized access attempts that may indicate a threat from an external source.- **Unauthenticated Logins:** The user attempting to log in via SSH (`seed@domain.com`) could be an example of an unsecured login attempt.- **Malicious Payloads/Brute Force Attacks:** The malware is likely part of an attack on the system through brute force, which would have been initiated by a successful SSH login from `seed@domain.com`.- **Authentication and Authorization:**  - No direct authentication attempts were made against the user account (`seed@domain.com`).  - External access or unauthorized access are not indicated within the network flow.  It is important to continue monitoring and reviewing the logs for any additional suspicious activities that may indicate an external breach."

In [23]:

generated_python_code = r'''
import subprocess
import re
from collections import Counter

result = subprocess.run(
    ["journalctl", "-u", "ssh", "--no-pager", "-n", "200"],
    capture_output=True,
    text=True
)

log = result.stdout

ips = re.findall(
    r"Failed password.*from ([0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)",
    log
)

counts = Counter(ips)

print("FAILED SSH LOGIN REPORT")
print("-----------------------")

if not counts:
    print("No failed SSH logins found. Claude AI")

for ip, count in counts.most_common():
    print(f"{ip}: {count} failed attempts. Claude AI")

    if count >= 3:
        print(f"  WARNING: {ip} may be suspicious. Claude AI")
'''


In [24]:

remote_file = "/tmp/analyze_ClaudeAI_ssh.py"

sftp = ssh.open_sftp()

with sftp.file(remote_file, "w") as f:
    f.write(generated_python_code)

sftp.close()

print("\nPython script uploaded to:")
print(remote_file)



Python script uploaded to:
/tmp/analyze_ClaudeAI_ssh.py


In [25]:



stdin, stdout, stderr = ssh.exec_command(
    f"python3 {remote_file}"
)

script_output = stdout.read().decode()
script_error = stderr.read().decode()

print("\n================ AGENT SCRIPT RESULT ================\n")
print(script_output)

if script_error:
    print("\nERROR:")
    print(script_error)





================ AGENT SCRIPT RESULT ================

FAILED SSH LOGIN REPORT
-----------------------
No failed SSH logins found. Claude AI



In [26]:



ssh.close()

print("\nSSH connection closed.")





SSH connection closed.
